# Natural language Processing with Recurrent Neural Networks 

In [48]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import keras_hub
from tensorflow import keras
from keras.models import Sequential
from keras.layers import Input,Embedding, GRU, Dense 

# IMDB dataset

See ***https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews*** for descritpion of this dataset

In [2]:
# unzip file 'IMDB Dataset.zip'to get 'IMDB Dataset.csv'

df = pd.read_csv('IMDB Dataset.csv')
x_data = df['review']       # Reviews/Input
y_data = df['sentiment']    # Sentiment/Output

# ENCODE SENTIMENT -> 0 & 1
y_data = y_data.replace('positive', 1)
y_data = y_data.replace('negative', 0)

/tmp/ipykernel_28740/647946610.py:9: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_data = y_data.replace('negative', 0)


In [3]:
# read first movie review
x_data[0]

"One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me about Oz was its brutality and unflinching scenes of violence, which set in right from the word GO. Trust me, this is not a show for the faint hearted or timid. This show pulls no punches with regards to drugs, sex or violence. Its is hardcore, in the classic use of the word.<br /><br />It is called OZ as that is the nickname given to the Oswald Maximum Security State Penitentary. It focuses mainly on Emerald City, an experimental section of the prison where all the cells have glass fronts and face inwards, so privacy is not high on the agenda. Em City is home to many..Aryans, Muslims, gangstas, Latinos, Christians, Italians, Irish and more....so scuffles, death stares, dodgy dealings and shady agreements are never far away.<br /><br />I would say the main appeal of the show is due to the fa

In [4]:
# label (sentiment) of this movie review
y_data[0] # positive review = 1

np.int64(1)

## In order to ***train a Recurrent Neural Network to Classify the movie reviews*** as 'positive' or 'negative' we decompose the IMDB dataset into two subsets: trainVal and test

### We used the command
 ## ***x_trainVal, x_test, y_trainVal, y_test = train_test_split(x_data, y_data, test_size = 0.2)***
 ### and  the resulting datasets were saved in the files 'imdb_x_trainVal.csv', 'imdb_y_trainVal.csv', 'imdb_x_test.csv' and 'imdb_y_test.csv'

In [5]:
x_trainVal = pd.read_csv('imdb_x_trainVal.csv')
y_trainVal = pd.read_csv('imdb_y_trainVal.csv')
x_test = pd.read_csv('imdb_x_test.csv')
y_test = pd.read_csv('imdb_y_test.csv')

# Employing one ***'tokenizer'***, every movie review is decomposed into a token sequence, which is subsequently transformed into an integer sequence.

In [9]:
tokenizer = keras_hub.models.GPT2Tokenizer.from_preset("gpt2_base_en")
# how many tokens in the vocabular
total_words = tokenizer.vocabulary_size()
print(total_words)

50257


## The following example illustrates the functionality of a tokenizer.

In [10]:
# Example text
text = 'Master Yoda has many wise sayings, but one of the most quoted would be this one. While training on Degobah in Empire Strikes Back ...'

# Convert the text above into a sequence of integers based on the token decomposition.
ord_tokens = tokenizer(text)
print(ord_tokens)

tf.Tensor(
[18254   575 11329   468   867 10787   910   654    11   475   530   286
   262   749 10947   561   307   428   530    13  2893  3047   319 25905
   672   993   287  8065 37836  5157  2644], shape=(31,), dtype=int32)


In [11]:
# print the sequence of tokens
[tokenizer.id_to_token(i).replace('Ġ', ' ') for i in ord_tokens]

['Master',
 ' Y',
 'oda',
 ' has',
 ' many',
 ' wise',
 ' say',
 'ings',
 ',',
 ' but',
 ' one',
 ' of',
 ' the',
 ' most',
 ' quoted',
 ' would',
 ' be',
 ' this',
 ' one',
 '.',
 ' While',
 ' training',
 ' on',
 ' Deg',
 'ob',
 'ah',
 ' in',
 ' Empire',
 ' Strikes',
 ' Back',
 ' ...']

# Convert 'x_trainVal' and 'x_test' into integer sequences for input into a Recurrent Neural Network (RNN).

In [12]:
# trainVal dataset is big 
len(x_trainVal)

40000

In [23]:
# We look at the integer sequence length for the first 10 reviews
experseq = tokenizer(x_trainVal['review'][:10])
[len(i) for i in experseq]

[186, 191, 70, 233, 761, 403, 62, 117, 364, 148]

### The resulting sequence of tokens have variable lengths and some are long.
### To accelerate training, we will truncate sequences to a maximum length of 200 terms.

In [25]:
k = 200
for list in experseq:
        if len(list)>k:
            del[list[k:]]

In [26]:
[len(i) for i in experseq]

[186, 191, 70, 200, 200, 200, 62, 117, 200, 148]

###  To simplify the training syntax by ensuring all input sequences have the same length, we will pad sequences shorter than 200 with zeros at the beginning.

In [27]:
k = 200
for list in experseq:
        if len(list)<k:
            aux = len(list)
            list[:0]= (k-aux)*[0]

In [28]:
[len(i) for i in experseq]

[200, 200, 200, 200, 200, 200, 200, 200, 200, 200]

In [31]:
# Now we can collect our sequences in a numpy array
array = np.array( experseq)
array.shape

(10, 200)

## As the above procedure would take long in the all x_trainVal dataset we di it once and saved the result on a numpy array. We did the the same with the x_test dataset. We can now just reload from the npy files we created. We also saved the corresponding labels in npy files

In [35]:
xtrainVal = np.load('imdbXtrainVal200seqs.npy')
ytrainVal = np.load('imdbLabelsTrainVal.npy')
print(xtrainVal.shape)
print(ytrainVal.shape)  
xtest = np.load('imdbXtest200seqs.npy')
ytest = np.load('imdbLabelsTest.npy')
print(xtest.shape)
print(ytest.shape) 

(40000, 200)
(40000,)
(10000, 200)
(10000,)


### We also decompose the trainVal datasets into train and validation datasets in order to train our model (neural network) 

In [36]:
xtrain = xtrainVal[:-500]
ytrain = ytrainVal[:-500]
xval  = xtrainVal[-500:]
yval = ytrainVal[-500:]

# Exercise: Build and train a recurrent neural network on the IMDB train dataset(using the validation data).Evaluate your model on the test set.